In [2]:
import pandas as pd
df =pd.read_csv('BlockData_2020_2025.csv')
df["time"] = pd.to_datetime(df["time"], dayfirst=True)


In [3]:
## spliting in difficulty adjustments


df['epoch']=df['id']//2016


In [ ]:
#df.head()

In [4]:
epoch_stats = df.groupby('epoch').agg({
    'block_time_secs': ['var', 'mean']
})

epoch_stats.columns = ['block_time_var', 'block_time_mean']
epoch_stats = epoch_stats.reset_index()

In [5]:
epoch_diff = df.groupby('epoch')['difficulty'].last().reset_index()

In [6]:
epoch_diff['diff_pct_change'] = epoch_diff['difficulty'].pct_change()
epoch_diff['shock_magnitude'] = epoch_diff['diff_pct_change'].abs()

In [7]:
epoch_time = df.groupby('epoch')['time'].last().reset_index()

In [8]:
data = epoch_stats.merge(epoch_diff, on='epoch')
data = data.merge(epoch_time, on='epoch')
data = data.dropna()

In [9]:
threshold = 2 * data['shock_magnitude'].std()

print(threshold)

data['large_shock'] = (data['shock_magnitude'] > threshold).astype(int)

0.08464335350979593


In [10]:
cutoff_date = pd.Timestamp('2021-07-01')  # around enforcement
cutoff_date_2023 = pd.Timestamp('2023-01-01')  # start of 2023
data['post_ban'] = (data['time'] >= cutoff_date).astype(int)
data['post_2023'] = (data['time'] >= cutoff_date_2023).astype(int)

In [18]:
import statsmodels.api as sm

X = data[['shock_magnitude']]
X = sm.add_constant(X)

y = data['block_time_var']

model = sm.OLS(y, X).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:         block_time_var   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     1.526
Date:                Wed, 06 May 2026   Prob (F-statistic):              0.219
Time:                        20:14:13   Log-Likelihood:                -2596.6
No. Observations:                 159   AIC:                             5197.
Df Residuals:                     157   BIC:                             5203.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            9.927e+05   3.48e+05      2.853      0.005    3.05e+05    1.68e+06
shock_magnitude -6.994e+06   5.66e+06     -1.235      0.219   -1.82e+07    4.19e+06
==============================================================================
Omnibus:                      300.280   Durbin-Watson:                   1.172
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            61102.356
Skew:                           9.535   Prob(JB):                         0.00
Kurtosis:                      97.124   Cond. No.                         23.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [16]:
X = data[['shock_magnitude', 'post_ban', 'post_2023']]
X['interaction'] = X['shock_magnitude'] * X['post_ban']

X = sm.add_constant(X)

model1 = sm.OLS(y, X).fit()

In [17]:
model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                log_var   R-squared:                       0.018
Model:                            OLS   Adj. R-squared:                 -0.008
Method:                 Least Squares   F-statistic:                    0.6941
Date:                Wed, 06 May 2026   Prob (F-statistic):              0.597
Time:                        20:14:08   Log-Likelihood:                -113.64
No. Observations:                 159   AIC:                             237.3
Df Residuals:                     154   BIC:                             252.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              12.7519      0.122    104.787      0.000      12.512      12.992
shock_magnitude     0.4554      1.475      0.309      0.758      -2.459       3.369
post_ban            0.0882      0.158      0.559      0.577      -0.224       0.400
post_2023           0.0763      0.098      0.776      0.439      -0.118       0.270
interaction        -2.0188      1.966     -1.027      0.306      -5.902       1.864
==============================================================================
Omnibus:                      260.121   Durbin-Watson:                   1.175
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            25816.597
Skew:                           7.497   Prob(JB):                         0.00
Kurtosis:                      63.597   Cond. No.                         82.3
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [14]:
import numpy as np

data['log_var'] = np.log(data['block_time_var'])

In [15]:
y = data['log_var']

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.ar_model import AutoReg

series = data['shock_magnitude']

# Check for missing values
print(f"Series shape: {series.shape}")
print(f"Missing values: {series.isna().sum()}")
print(f"Series type: {type(series)}")

# Clean the data
series_clean = series.dropna()

# Then refit with clean data
arima_model = ARIMA(series_clean, order=(1, 0, 0)).fit()
ar1_model = AutoReg(series_clean, lags=1, old_names=False).fit()

print(arima_model.summary())
print(ar1_model.summary())

                               SARIMAX Results                                
Dep. Variable:        shock_magnitude   No. Observations:                  159
Model:                 ARIMA(1, 0, 0)   Log Likelihood                 278.383
Date:                Wed, 06 May 2026   AIC                           -550.766
Time:                        20:25:30   BIC                           -541.559
Sample:                             0   HQIC                          -547.027
                                - 159                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0447      0.005      8.710      0.000       0.035       0.055
ar.L1          0.0916      0.072      1.274      0.203      -0.049       0.232
sigma2         0.0018      0.000     12.583      0.0

ValueError: operands could not be broadcast together with shapes (158,) (159,) 